# 🌿 AgriScan AI: PlantVillage Two-Stage Hybrid Training Pipeline
### Stage 1: Deep Feature Extractor (ResNet-50) + Stage 2: Calibrated SVM Classifier
---
This notebook trains the deep convolutional feature extractor on the **PlantVillage foliar dataset** (54,305 leaf images, 38 classes), extracts 2,048-dimensional dense visual representations, fits an RBF Support Vector Machine, and exports `models/plantvillage_resnet50.pth`.

## 1. Setup Environment & Hardware Acceleration

In [ ]:
# Verify GPU acceleration
%nvidia-smi

# Install dependencies
%pip install -q torch torchvision torchaudio scikit-learn joblib kagglehub matplotlib seaborn

## 2. Download and Load PlantVillage Dataset

In [ ]:
import os
import kagglehub

# Download latest version of PlantVillage foliar dataset (54,305 images across 38 classes)
path = kagglehub.dataset_download("emmarex/plantdisease")
print("PlantVillage Dataset downloaded to:", path)

# Check class directories
dataset_dir = os.path.join(path, "PlantVillage") if os.path.exists(os.path.join(path, "PlantVillage")) else path
classes = sorted(os.listdir(dataset_dir))
print(f"Total Pathology Classes: {len(classes)}")
print("Sample Classes:", classes[:5])

## 3. Data Augmentation & PyTorch DataLoaders

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

IMG_SIZE = 224
BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Active Compute Device:", DEVICE)

# Augmentation for field robustness (flips, rotations, lighting variations)
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 80% Train, 10% Validation, 10% Test split
full_dataset = datasets.ImageFolder(root=dataset_dir)
total = len(full_dataset)
train_len = int(0.80 * total)
val_len = int(0.10 * total)
test_len = total - train_len - val_len

train_set, val_set, test_set = random_split(
    full_dataset, [train_len, val_len, test_len],
    generator=torch.Generator().manual_seed(42)
)

train_set.dataset.transform = train_transforms
val_set.dataset.transform = val_transforms
test_set.dataset.transform = val_transforms

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Dataset Partition: {train_len} Train | {val_len} Validation | {test_len} Test")

## 4. Stage 1: Fine-Tuning ResNet-50 Deep Feature Extractor

In [ ]:
import torch.nn as nn
from torchvision import models

# Load Pre-trained ResNet-50
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Fine-tune deep layers while freezing early convolutions
for param in list(model.parameters())[:-15]:
    param.requires_grad = False

in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, len(classes))
)
model = model.to(DEVICE)

# Loss function with label smoothing to prevent overconfident cross-species errors
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

# Training Loop (15 Epochs)
EPOCHS = 15
best_acc = 0.0
os.makedirs("models", exist_ok=True)

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    scheduler.step()
    train_acc = correct / total

    # Validation Loop
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == targets).sum().item()
            val_total += targets.size(0)

    val_acc = val_correct / val_total
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {running_loss/total:.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "models/plantvillage_resnet50.pth")
        print(f"--> Checkpoint saved: {val_acc*100:.2f}%")

## 5. Stage 2: Feature Embedding Extraction (2,048-Dim) & Support Vector Machine

In [ ]:
import numpy as np
import joblib
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# Strip classification head to use ResNet-50 as pure 2,048-dim feature extractor
model.load_state_dict(torch.load("models/plantvillage_resnet50.pth"))
model.eval()
feature_extractor = nn.Sequential(*list(model.children())[:-1]).to(DEVICE)

def extract_vectors(dataloader):
    feats, labs = [], []
    with torch.no_grad():
        for imgs, targets in dataloader:
            imgs = imgs.to(DEVICE)
            out = feature_extractor(imgs)
            out = torch.flatten(out, 1)  # (batch, 2048)
            feats.append(out.cpu().numpy())
            labs.append(targets.numpy())
    return np.vstack(feats), np.concatenate(labs)

print("Extracting 2,048-dimensional feature vectors...")
X_train, y_train = extract_vectors(train_loader)
X_test, y_test = extract_vectors(test_loader)

# Fit Calibrated SVM Classifier
print("Training RBF Support Vector Machine on visual embeddings...")
svm_classifier = SVC(C=1.0, kernel='rbf', probability=True, random_state=42, cache_size=1000)
svm_classifier.fit(X_train, y_train)

# Evaluation
y_pred = svm_classifier.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\n🏆 Final PlantVillage Hybrid Test Accuracy: {acc * 100:.2f}%")
print("\nClassification Report Summary:")
print(classification_report(y_test, y_pred, target_names=classes[:len(set(y_test))]))

# ── Save Trained SVM Model to Disk ──────────────────────────────────────────
OUTPUT_SVM_PATH = 'models/plantvillage_svm_model.pkl'
os.makedirs('models', exist_ok=True)
joblib.dump(svm_classifier, OUTPUT_SVM_PATH)
svm_mb = os.path.getsize(OUTPUT_SVM_PATH) / (1024 * 1024)
print(f'[*] Saved trained RBF-SVM model to: {OUTPUT_SVM_PATH} ({svm_mb:.2f} MB)')


## 6. Export Final Weights for Flask Web Deployment

In [ ]:
# Confirm both production weights are saved
resnet_mb = os.path.getsize('models/plantvillage_resnet50.pth') / (1024 * 1024)
svm_mb = os.path.getsize('models/plantvillage_svm_model.pkl') / (1024 * 1024)
print(f'[*] Production ResNet-50 weights : models/plantvillage_resnet50.pth ({resnet_mb:.2f} MB)')
print(f'[*] Production RBF-SVM model     : models/plantvillage_svm_model.pkl ({svm_mb:.2f} MB)')
print('[*] Done! Both model artifacts are ready for the Flask deployment application.')
